###  Imports & Config

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime


SILVER_NAME = "Silver_Lakehouse"
GOLD_NAME = "Gold_Lakehouse"

SILVER_PATH = f"abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo"
GOLD_PATH = f"abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/da923130-af44-4b41-9efd-b133ac83e6b7/Tables/dbo"

def read_silver(table_name):
    return spark.read.format("delta").load(f"{SILVER_PATH}/{table_name}")

def read_gold(table_name):
    return spark.read.format("delta").load(f"{GOLD_PATH}/{table_name}")

def gold_table_exists(table_name):
    try:
        spark.read.format("delta").load(f"{GOLD_PATH}/{table_name}")
        return True
    except:
        return False

def write_to_gold(df, table_name):
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)
    print(f"✅ {table_name} → {df.count():,} rows written to Gold")

print("✅ Config loaded")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 3, Finished, Available, Finished, False)

✅ Config loaded


### Read All Silver Tables

In [2]:
silver_customers = read_silver("customers")
silver_orders = read_silver("orders")
silver_order_items = read_silver("order_items")
silver_products = read_silver("products")
silver_product_reviews = read_silver("product_reviews")

print(f"📥 Customers:       {silver_customers.count():,}")
print(f"📥 Orders:          {silver_orders.count():,}")
print(f"📥 Order Items:     {silver_order_items.count():,}")
print(f"📥 Products:        {silver_products.count():,}")
print(f"📥 Product Reviews: {silver_product_reviews.count():,}")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 4, Finished, Available, Finished, False)

📥 Customers:       1,048,724
📥 Orders:          8,000,912
📥 Order Items:     20,002,735
📥 Products:        20,000
📥 Product Reviews: 4,000,000


### DIM_DATE (Incremental — Add New Dates Only)

In [ ]:
print("=" * 60)
print("📅 PROCESSING: DIM_DATE")
print("=" * 60)

# Generate full date spine
df_dim_date = (
    spark.sql("""
        SELECT explode(sequence(
            to_date('2021-08-31'),
            to_date('2026-12-31'),
            interval 1 day
        )) AS full_date
    """)
    .withColumn("date_key", F.date_format(F.col("full_date"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month_number", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend",
        F.when(F.dayofweek("full_date").isin(1, 7), F.lit(True))
         .otherwise(F.lit(False))
    )
    .select("date_key", "full_date", "year", "quarter",
            "month_number", "month_name", "day_of_week",
            "day_name", "is_weekend")
)


write_to_gold(df_dim_date, "dim_date")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 5, Finished, Available, Finished, False)

📅 PROCESSING: DIM_DATE
✅ dim_date → 1,949 rows written to Gold


###  DIM_CUSTOMER (Incremental MERGE)

In [ ]:
print("=" * 60)
print("👤 PROCESSING: DIM_CUSTOMER")
print("=" * 60)

w = Window.orderBy("customer_id")

df_dim_customer_new = (silver_customers
    .withColumn("customer_key", F.row_number().over(w))
    .withColumn("signup_date", F.col("start_date"))
    .withColumn("is_current",
        F.when(F.col("end_date").isNull(), F.lit(True))
         .otherwise(F.lit(False))
    )
    .select(
        "customer_key", "customer_id", "name", "email",
        "gender", "country", "signup_date", "end_date", "is_current"
    )
)

# Add unknown member
unknown_customer = spark.sql("""
    SELECT 
        CAST(-1 AS INT) AS customer_key,
        CAST(-1 AS INT) AS customer_id,
        'Unknown' AS name,
        'unknown' AS email,
        'Unknown' AS gender,
        'Unknown' AS country,
        CAST(NULL AS DATE) AS signup_date,
        CAST(NULL AS DATE) AS end_date,
        CAST(TRUE AS BOOLEAN) AS is_current
""")

df_dim_customer = df_dim_customer_new.union(unknown_customer)

write_to_gold(df_dim_customer, "dim_customer")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 6, Finished, Available, Finished, False)

👤 PROCESSING: DIM_CUSTOMER
✅ dim_customer → 1,048,725 rows written to Gold


### DIM_PRODUCT (SCD Type 2 for Price Changes)

In [5]:
print("=" * 60)
print("🏷️ PROCESSING: DIM_PRODUCT (SCD Type 2)")
print("=" * 60)

product_path = f"{GOLD_PATH}/dim_product"

if gold_table_exists("dim_product"):
    # --- SCD Type 2: Handle price changes ---
    df_existing_dim = read_gold("dim_product").filter(F.col("product_id") != -1)
    
    # Get current products from Silver
    df_silver_products = silver_products.select(
        "product_id", "product_name", "category", "brand", "price"
    )
    
    # Find products where price changed
    df_price_changed = (df_existing_dim
        .filter(F.col("is_current") == True)
        .join(df_silver_products, on="product_id", how="inner")
        .filter(df_existing_dim.price != df_silver_products.price)
        .select(df_existing_dim.product_id)
    )
    
    price_changed_count = df_price_changed.count()
    
    if price_changed_count > 0:
        print(f"   💰 {price_changed_count} products have price changes — applying SCD Type 2")
        
        # Expire old rows
        delta_product = DeltaTable.forPath(spark, product_path)
        
        (delta_product.alias("target")
            .merge(
                df_price_changed.alias("source"),
                "target.product_id = source.product_id AND target.is_current = true"
            )
            .whenMatchedUpdate(set={
                "is_current": F.lit(False),
                "effective_price_end_date": F.current_date()
            })
            .execute()
        )
        print(f"   ✅ Expired {price_changed_count} old price rows")
        
        # Insert new rows with updated prices
        max_key = read_gold("dim_product").agg(F.max("product_key")).collect()[0][0]
        w_new = Window.orderBy("product_id")
        
        df_new_price_rows = (df_silver_products
            .join(df_price_changed, on="product_id", how="inner")
            .withColumn("product_key", F.lit(max_key) + F.row_number().over(w_new))
            .withColumn("effective_price_start_date", F.current_date())
            .withColumn("effective_price_end_date", F.lit(None).cast(DateType()))
            .withColumn("is_current", F.lit(True))
            .select(
                "product_key", "product_id", "product_name", "category",
                "brand", "price", "effective_price_start_date",
                "effective_price_end_date", "is_current"
            )
        )
        
        df_new_price_rows.write.format("delta").mode("append").save(product_path)
        print(f"   ✅ Inserted {df_new_price_rows.count()} new price rows")
    else:
        print("   ✅ No price changes detected")
    
    # Add any NEW products (not in dim yet)
    existing_product_ids = df_existing_dim.select("product_id").distinct()
    df_truly_new = df_silver_products.join(existing_product_ids, on="product_id", how="left_anti")
    
    if df_truly_new.count() > 0:
        max_key = read_gold("dim_product").agg(F.max("product_key")).collect()[0][0]
        w_new = Window.orderBy("product_id")
        
        df_new_products = (df_truly_new
            .withColumn("product_key", F.lit(max_key) + F.row_number().over(w_new))
            .withColumn("effective_price_start_date", F.current_date())
            .withColumn("effective_price_end_date", F.lit(None).cast(DateType()))
            .withColumn("is_current", F.lit(True))
            .select(
                "product_key", "product_id", "product_name", "category",
                "brand", "price", "effective_price_start_date",
                "effective_price_end_date", "is_current"
            )
        )
        
        df_new_products.write.format("delta").mode("append").save(product_path)
        print(f"   ✅ Added {df_new_products.count()} new products")
    
    final_count = read_gold("dim_product").count()
    print(f"   📊 Total dim_product rows: {final_count:,}")

else:
    # First run — create from scratch
    w = Window.orderBy("product_id")
    
    df_dim_product = (silver_products
        .withColumn("product_key", F.row_number().over(w))
        .withColumn("effective_price_start_date", F.current_date())
        .withColumn("effective_price_end_date", F.lit(None).cast(DateType()))
        .withColumn("is_current", F.lit(True))
        .select(
            "product_key", "product_id", "product_name", "category",
            "brand", "price", "effective_price_start_date",
            "effective_price_end_date", "is_current"
        )
    )
    
    unknown_product = spark.sql("""
        SELECT
            CAST(-1 AS INT) AS product_key,
            CAST(-1 AS INT) AS product_id,
            'Unknown' AS product_name,
            'Unknown' AS category,
            'Unknown' AS brand,
            CAST(NULL AS DECIMAL(10,2)) AS price,
            CAST(NULL AS DATE) AS effective_price_start_date,
            CAST(NULL AS DATE) AS effective_price_end_date,
            CAST(TRUE AS BOOLEAN) AS is_current
    """)
    
    df_dim_product = df_dim_product.union(unknown_product)
    write_to_gold(df_dim_product, "dim_product")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 7, Finished, Available, Finished, False)

🏷️ PROCESSING: DIM_PRODUCT (SCD Type 2)
   💰 1 products have price changes — applying SCD Type 2
   ✅ Expired 1 old price rows
   ✅ Inserted 0 new price rows
   📊 Total dim_product rows: 20,001


###  FACT_SALES (Full Rebuild)

In [6]:
print("=" * 60)
print("💰 PROCESSING: FACT_SALES")
print("=" * 60)

# Read current dimensions
df_dim_customer = read_gold("dim_customer")
df_dim_product = read_gold("dim_product")

# Join orders + order items
df_sales_base = silver_order_items.join(silver_orders, on="order_id", how="inner")

# Lookup customer_key (left join + fill -1)
df_sales_base = (df_sales_base
    .join(
        df_dim_customer.filter(F.col("customer_key") != -1)
            .select("customer_key", "customer_id"),
        on="customer_id", how="left"
    )
    .withColumn("customer_key",
        F.when(F.col("customer_key").isNull(), F.lit(-1))
         .otherwise(F.col("customer_key"))
    )
)

# Lookup product_key — match on CURRENT price only
df_sales_base = (df_sales_base
    .join(
        df_dim_product
            .filter((F.col("product_key") != -1) & (F.col("is_current") == True))
            .select("product_key", "product_id"),
        on="product_id", how="left"
    )
    .withColumn("product_key",
        F.when(F.col("product_key").isNull(), F.lit(-1))
         .otherwise(F.col("product_key"))
    )
)

# Create date key
df_fact_sales = (df_sales_base
    .withColumn("order_date_key",
        F.date_format(F.col("order_date"), "yyyyMMdd").cast(IntegerType())
    )
    .select(
        "order_item_id", "customer_key", "product_key",
        "order_date_key", "order_id", "quantity", "unit_price",
        "line_total", "payment_method", "shipping_country"
    )
    .withColumnRenamed("line_total", "total_amount")
)

write_to_gold(df_fact_sales, "fact_sales")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 8, Finished, Available, Finished, False)

💰 PROCESSING: FACT_SALES
✅ fact_sales → 20,002,735 rows written to Gold


### FACT_REVIEWS (Full Rebuild)

In [7]:
print("=" * 60)
print("⭐ PROCESSING: FACT_REVIEWS")
print("=" * 60)

df_dim_customer = read_gold("dim_customer")
df_dim_product = read_gold("dim_product")

# Lookup customer_key
df_reviews_base = (silver_product_reviews
    .join(
        df_dim_customer.filter(F.col("customer_key") != -1)
            .select("customer_key", "customer_id"),
        on="customer_id", how="left"
    )
    .withColumn("customer_key",
        F.when(F.col("customer_key").isNull(), F.lit(-1))
         .otherwise(F.col("customer_key"))
    )
)

# Lookup product_key
df_reviews_base = (df_reviews_base
    .join(
        df_dim_product
            .filter((F.col("product_key") != -1) & (F.col("is_current") == True))
            .select("product_key", "product_id"),
        on="product_id", how="left"
    )
    .withColumn("product_key",
        F.when(F.col("product_key").isNull(), F.lit(-1))
         .otherwise(F.col("product_key"))
    )
)

# Create date key
df_fact_reviews = (df_reviews_base
    .withColumn("review_date_key",
        F.date_format(F.col("review_date"), "yyyyMMdd").cast(IntegerType())
    )
    .select(
        "review_id", "customer_key", "product_key",
        "review_date_key", "rating", "review_text"
    )
)

write_to_gold(df_fact_reviews, "fact_reviews")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 9, Finished, Available, Finished, False)

⭐ PROCESSING: FACT_REVIEWS
✅ fact_reviews → 4,000,000 rows written to Gold


### AGG_CUSTOMER_360 (Full Recalculate)

In [8]:
print("=" * 60)
print("💰 PROCESSING: AGG_CUSTOMER_360")
print("=" * 60)

df_dim_customer = read_gold("dim_customer")
df_dim_date = read_gold("dim_date")
df_fact_sales = read_gold("fact_sales")
df_fact_reviews = read_gold("fact_reviews")

# Sales metrics per customer
customer_sales = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .groupBy("customer_key")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_items_bought"),
        F.sum("total_amount").alias("total_lifetime_spend"),
        F.min("full_date").alias("first_order_date"),
        F.max("full_date").alias("last_order_date")
    )
    .withColumn("avg_order_value",
        F.round(F.col("total_lifetime_spend") / F.col("total_orders"), 2)
    )
    .withColumn("days_since_last_order",
        F.datediff(F.current_date(), F.col("last_order_date"))
    )
)

# Review metrics per customer
customer_reviews = (df_fact_reviews
    .groupBy("customer_key")
    .agg(
        F.count("review_id").alias("total_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating_given")
    )
)

# Combine
df_agg_customer = (df_dim_customer
    .join(customer_sales, on="customer_key", how="left")
    .join(customer_reviews, on="customer_key", how="left")
    .fillna(0, subset=["total_orders", "total_items_bought", "total_lifetime_spend", "total_reviews"])
)

# Customer segment (percentile-based)
w_spend = Window.orderBy(F.col("total_lifetime_spend").desc())
df_agg_customer = df_agg_customer.withColumn("spend_rank_pct", F.percent_rank().over(w_spend))

df_agg_customer = df_agg_customer.withColumn(
    "customer_segment",
    F.when(F.col("total_orders") == 0, F.lit("Inactive"))
     .when(F.col("spend_rank_pct") <= 0.10, F.lit("Platinum"))
     .when(F.col("spend_rank_pct") <= 0.30, F.lit("Gold"))
     .when(F.col("spend_rank_pct") <= 0.60, F.lit("Silver"))
     .otherwise(F.lit("Bronze"))
)

# Customer status (recency-based)
df_agg_customer = df_agg_customer.withColumn(
    "customer_status",
    F.when(F.col("total_orders") == 0, F.lit("Never Purchased"))
     .when(F.col("days_since_last_order") <= 90, F.lit("Active"))
     .when(F.col("days_since_last_order") <= 180, F.lit("At Risk"))
     .otherwise(F.lit("Churned"))
)

df_agg_customer_360 = df_agg_customer.select(
    "customer_key", "customer_id", "name", "country", "gender", "signup_date",
    "total_orders", "total_items_bought",
    F.round("total_lifetime_spend", 2).alias("total_lifetime_spend"),
    F.round("avg_order_value", 2).alias("avg_order_value"),
    "first_order_date", "last_order_date", "days_since_last_order",
    "total_reviews", "avg_rating_given",
    "customer_segment", "customer_status"
)

write_to_gold(df_agg_customer_360, "agg_customer_360")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 10, Finished, Available, Finished, False)

💰 PROCESSING: AGG_CUSTOMER_360
✅ agg_customer_360 → 1,048,725 rows written to Gold


### AGG_PRODUCT_SCORECARD (Full Recalculate)

In [9]:
print("=" * 60)
print("📦 PROCESSING: AGG_PRODUCT_SCORECARD")
print("=" * 60)

df_dim_product = read_gold("dim_product").filter(F.col("is_current") == True)
df_fact_sales = read_gold("fact_sales")
df_fact_reviews = read_gold("fact_reviews")

product_sales = (df_fact_sales
    .groupBy("product_key")
    .agg(
        F.sum("quantity").alias("total_units_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.countDistinct("customer_key").alias("unique_buyers")
    )
)

product_reviews = (df_fact_reviews
    .groupBy("product_key")
    .agg(
        F.count("review_id").alias("total_reviews"),
        F.round(F.avg("rating"), 2).alias("avg_rating")
    )
)

df_agg_product = (df_dim_product
    .select("product_key", "product_id", "product_name", "category", "brand", "price")
    .join(product_sales, on="product_key", how="left")
    .join(product_reviews, on="product_key", how="left")
    .fillna(0, subset=["total_units_sold", "total_revenue", "total_orders", "unique_buyers", "total_reviews"])
    .fillna(0.0, subset=["avg_rating"])
)

df_agg_product = df_agg_product.withColumn(
    "review_to_purchase_ratio",
    F.when(F.col("unique_buyers") > 0,
        F.round(F.col("total_reviews") / F.col("unique_buyers"), 2)
    ).otherwise(F.lit(0.0))
)

median_revenue = df_agg_product.filter(F.col("total_revenue") > 0) \
    .approxQuantile("total_revenue", [0.5], 0.01)[0]

df_agg_product = df_agg_product.withColumn(
    "product_tier",
    F.when(F.col("total_revenue") == 0, F.lit("No Sales"))
     .when((F.col("total_revenue") >= median_revenue) & (F.col("avg_rating") > 3.5), F.lit("⭐ Star Product"))
     .when((F.col("total_revenue") >= median_revenue) & (F.col("avg_rating") <= 3.5), F.lit("🐄 Cash Cow"))
     .when((F.col("total_revenue") < median_revenue) & (F.col("avg_rating") > 3.5), F.lit("💎 Hidden Gem"))
     .otherwise(F.lit("❌ Underperformer"))
)

df_agg_product_scorecard = df_agg_product.select(
    "product_key", "product_id", "product_name", "category", "brand", "price",
    "total_units_sold", F.round("total_revenue", 2).alias("total_revenue"),
    "total_orders", "unique_buyers", "total_reviews", "avg_rating",
    "review_to_purchase_ratio", "product_tier"
)

write_to_gold(df_agg_product_scorecard, "agg_product_scorecard")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 11, Finished, Available, Finished, False)

📦 PROCESSING: AGG_PRODUCT_SCORECARD
✅ agg_product_scorecard → 19,997 rows written to Gold


### AGG_GEOGRAPHIC_PERFORMANCE (Full Recalculate)

In [10]:
print("=" * 60)
print("🌍 PROCESSING: AGG_GEOGRAPHIC_PERFORMANCE")
print("=" * 60)

df_dim_product = read_gold("dim_product").filter(F.col("is_current") == True)
df_dim_customer = read_gold("dim_customer")
df_fact_sales = read_gold("fact_sales")
df_fact_reviews = read_gold("fact_reviews")

geo_base = (df_fact_sales
    .groupBy("shipping_country")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("customer_key").alias("total_customers"),
        F.sum("quantity").alias("total_items_sold")
    )
    .withColumn("avg_order_value", F.round(F.col("total_revenue") / F.col("total_orders"), 2))
)

# Top category per country
cat_rev = (df_fact_sales
    .join(df_dim_product.select("product_key", "category"), on="product_key", how="inner")
    .groupBy("shipping_country", "category")
    .agg(F.sum("total_amount").alias("cat_revenue"))
)
w_cat = Window.partitionBy("shipping_country").orderBy(F.col("cat_revenue").desc())
top_cat = cat_rev.withColumn("rn", F.row_number().over(w_cat)).filter("rn = 1").select("shipping_country", F.col("category").alias("top_category"))

# Top brand per country
brand_rev = (df_fact_sales
    .join(df_dim_product.select("product_key", "brand"), on="product_key", how="inner")
    .groupBy("shipping_country", "brand")
    .agg(F.sum("total_amount").alias("brand_revenue"))
)
w_brand = Window.partitionBy("shipping_country").orderBy(F.col("brand_revenue").desc())
top_brand = brand_rev.withColumn("rn", F.row_number().over(w_brand)).filter("rn = 1").select("shipping_country", F.col("brand").alias("top_brand"))

# Top payment method per country
pm_counts = (df_fact_sales
    .groupBy("shipping_country", "payment_method")
    .agg(F.countDistinct("order_id").alias("pm_orders"))
)
w_pm = Window.partitionBy("shipping_country").orderBy(F.col("pm_orders").desc())
top_pm = pm_counts.withColumn("rn", F.row_number().over(w_pm)).filter("rn = 1").select("shipping_country", F.col("payment_method").alias("top_payment_method"))

# Avg rating per country
geo_ratings = (df_fact_reviews
    .join(df_dim_customer.select("customer_key", "country"), on="customer_key", how="inner")
    .groupBy("country")
    .agg(F.round(F.avg("rating"), 2).alias("avg_product_rating"))
)

# Combine
df_agg_geo = (geo_base
    .join(top_cat, on="shipping_country", how="left")
    .join(top_brand, on="shipping_country", how="left")
    .join(top_pm, on="shipping_country", how="left")
    .join(geo_ratings, geo_base.shipping_country == geo_ratings.country, "left")
    .drop("country")
)

df_agg_geo = df_agg_geo.withColumn(
    "revenue_rank", F.row_number().over(Window.orderBy(F.col("total_revenue").desc()))
)

df_agg_geographic = df_agg_geo.select(
    "shipping_country", "total_orders",
    F.round("total_revenue", 2).alias("total_revenue"),
    "total_customers", "avg_order_value", "total_items_sold",
    "top_category", "top_brand", "top_payment_method",
    "avg_product_rating", "revenue_rank"
)

write_to_gold(df_agg_geographic, "agg_geographic_performance")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 12, Finished, Available, Finished, False)

🌍 PROCESSING: AGG_GEOGRAPHIC_PERFORMANCE
✅ agg_geographic_performance → 243 rows written to Gold


### AGG_MONTHLY_REVENUE_TRENDS (Full Recalculate)

In [11]:
print("=" * 60)
print("📅 PROCESSING: AGG_MONTHLY_REVENUE_TRENDS")
print("=" * 60)

df_dim_date = read_gold("dim_date")
df_dim_product = read_gold("dim_product").filter(F.col("is_current") == True)
df_fact_sales = read_gold("fact_sales")

# Monthly base
monthly_base = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date", "year", "month_number", "month_name"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .groupBy("year", "month_number", "month_name")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_items_sold"),
        F.countDistinct("customer_key").alias("unique_customers")
    )
    .withColumn("avg_order_value", F.round(F.col("total_revenue") / F.col("total_orders"), 2))
)

# New customers per month (cohort)
customer_first = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date", "year", "month_number"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .groupBy("customer_key")
    .agg(F.min("full_date").alias("first_order_date"))
    .withColumn("cohort_year", F.year("first_order_date"))
    .withColumn("cohort_month", F.month("first_order_date"))
)

new_cust_monthly = (customer_first
    .groupBy(F.col("cohort_year").alias("year"), F.col("cohort_month").alias("month_number"))
    .agg(F.count("customer_key").alias("new_customers"))
)

# Returning customers per month
all_cust_monthly = (df_fact_sales
    .join(df_dim_date.select("date_key", "full_date", "year", "month_number"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .select("customer_key", "year", "month_number").distinct()
)

returning = (all_cust_monthly
    .join(customer_first, on="customer_key", how="inner")
    .filter((F.col("year") != F.col("cohort_year")) | (F.col("month_number") != F.col("cohort_month")))
    .groupBy("year", "month_number")
    .agg(F.countDistinct("customer_key").alias("returning_customers"))
)

# Combine
df_monthly = (monthly_base
    .join(new_cust_monthly, on=["year", "month_number"], how="left")
    .join(returning, on=["year", "month_number"], how="left")
    .fillna(0, subset=["new_customers", "returning_customers"])
)

# Retention rate
df_monthly = df_monthly.withColumn(
    "retention_rate",
    F.when(F.col("unique_customers") > 0,
        F.round(F.col("returning_customers") / F.col("unique_customers") * 100, 2)
    ).otherwise(F.lit(0.0))
)

# Revenue growth MoM
w_month = Window.orderBy("year", "month_number")
df_monthly = df_monthly.withColumn("prev_revenue", F.lag("total_revenue").over(w_month))
df_monthly = df_monthly.withColumn(
    "revenue_growth_pct",
    F.when(F.col("prev_revenue").isNotNull() & (F.col("prev_revenue") > 0),
        F.round((F.col("total_revenue") - F.col("prev_revenue")) / F.col("prev_revenue") * 100, 2)
    ).otherwise(F.lit(None))
)

# Top category per month
mcat = (df_fact_sales
    .join(df_dim_date.select("date_key", "year", "month_number"),
          df_fact_sales.order_date_key == df_dim_date.date_key, "inner")
    .join(df_dim_product.select("product_key", "category"), on="product_key", how="inner")
    .groupBy("year", "month_number", "category")
    .agg(F.sum("total_amount").alias("cat_rev"))
)
w_mcat = Window.partitionBy("year", "month_number").orderBy(F.col("cat_rev").desc())
top_mcat = mcat.withColumn("rn", F.row_number().over(w_mcat)).filter("rn = 1").select("year", "month_number", F.col("category").alias("top_category"))

df_monthly = df_monthly.join(top_mcat, on=["year", "month_number"], how="left")

df_monthly_trends = df_monthly.select(
    "year", "month_number", "month_name",
    F.round("total_revenue", 2).alias("total_revenue"),
    "total_orders", "total_items_sold", "avg_order_value",
    "unique_customers", "new_customers", "returning_customers",
    "retention_rate", "revenue_growth_pct", "top_category"
).orderBy("year", "month_number")

write_to_gold(df_monthly_trends, "agg_monthly_revenue_trends")

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 13, Finished, Available, Finished, False)

📅 PROCESSING: AGG_MONTHLY_REVENUE_TRENDS
✅ agg_monthly_revenue_trends → 50 rows written to Gold


### Final Validation

In [12]:
print("=" * 60)
print("✅ GOLD LAYER VERIFICATION (POST-INCREMENT)")
print("=" * 60)

gold_tables = [
    "dim_date", "dim_customer", "dim_product",
    "fact_sales", "fact_reviews",
    "agg_customer_360", "agg_product_scorecard",
    "agg_geographic_performance", "agg_monthly_revenue_trends"
]

for table in gold_tables:
    try:
        df = spark.read.format("delta").table(table)
        print(f"\n📋 {table.upper()} — {df.count():,} rows, {len(df.columns)} cols")
        for field in df.schema.fields:
            nullable = "nullable" if field.nullable else "not null"
            print(f"   ├── {field.name:<30} {str(field.dataType):<20} {nullable}")
    except Exception as e:
        print(f"\n❌ {table.upper()} — ERROR: {str(e)[:100]}")

print("\n" + "=" * 60)
print("🎉 INCREMENTAL SILVER → GOLD COMPLETE!")
print("=" * 60)

StatementMeta(, 7c738807-e4e2-48ed-ba1b-8f4a970d00e2, 14, Finished, Available, Finished, False)

✅ GOLD LAYER VERIFICATION (POST-INCREMENT)

📋 DIM_DATE — 1,949 rows, 9 cols
   ├── date_key                       IntegerType()        nullable
   ├── full_date                      DateType()           nullable
   ├── year                           IntegerType()        nullable
   ├── quarter                        IntegerType()        nullable
   ├── month_number                   IntegerType()        nullable
   ├── month_name                     StringType()         nullable
   ├── day_of_week                    IntegerType()        nullable
   ├── day_name                       StringType()         nullable
   ├── is_weekend                     BooleanType()        nullable

📋 DIM_CUSTOMER — 1,048,725 rows, 9 cols
   ├── customer_key                   IntegerType()        nullable
   ├── customer_id                    IntegerType()        nullable
   ├── name                           StringType()         nullable
   ├── email                          StringType()         nullable